# Đánh giá toàn hệ thống trên Kaggle TPU v5e-8 (vLLM)

Hệ thống đầy đủ: PostgreSQL 16 + pgvector, Redis, Neo4j 5.18; 4 model chạy bằng **vLLM trên TPU**, mỗi model 2 chip:
LLM #1 `Qwen2.5-7B-Instruct`, LLM #2 `qwen-huhu` (LoRA gộp vào Qwen2.5-7B-Instruct, **bf16**), LLM #3 kiểm tra câu trả
lời cuối `gemma-3-12b-it`, giám khảo chấm điểm `gemma-3-4b-it` (không nằm trong hệ thống).
Chạy: crisis, retrieval, P/B1/B2/B3 và **P0 = hệ thống cũ** (baseline của hệ thống final) trên crisis + safety,
grounding và định tuyến nhiều lượt.

**Cài đặt notebook:** Accelerator = **TPU v5e-8**, Internet = **On**.

**Input (Add Input):**
1. Dataset `huhu-system-tpu` (file `huhu-system-tpu.zip` tạo bằng `python scripts/make_kaggle_package.py --tpu`; đã kèm LoRA adapter).
2. Models → **Gemma 3** → Transformers → `gemma-3-12b-it` và `gemma-3-4b-it` (cần đồng ý license Gemma trên Kaggle).
   Nếu không gắn, notebook tải từ Hugging Face bằng secret `HF_TOKEN` (tài khoản đã đồng ý license Gemma).
3. (Chỉ khi chạy tiếp) Output của lần chạy trước của notebook này: các ca đã xong được bỏ qua.

**Khác bản GPU (Ollama):** model chạy bf16 thay cho GGUF Q4_K_M, tham số sinh mặc định của vLLM thay cho Ollama
(cùng temperature), nên kết quả không so trực tiếp được với lần chạy bằng Ollama.

Thứ tự: chép code -> kiểm sha256 -> hạ tầng -> cài vLLM TPU -> tải/gộp model -> bật 4 server -> seed DB -> kiểm DB
-> unit test -> chạy thử -> crisis, retrieval -> P, B1, B2, B3 -> chấm -> grounding -> nén kết quả.
Bước kiểm tra nào lỗi thì notebook dừng (raise), không chạy test.

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, time
W = "/kaggle/working"
APP = f"{W}/AIM/Hybrid-database-main"
MODELS_DIR = "/tmp/models"          # ngoài /kaggle/working (giới hạn 20 GB) để chứa trọng số
VENV = f"{W}/vllm-venv"             # vllm-tpu cần Python 3.12, tách khỏi Python mặc định của Kaggle

def find(pattern):
    return sorted(p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

def sh(cmd, log=None):
    """Chạy lệnh shell trong thư mục app, in log trực tiếp; lỗi thì dừng notebook."""
    print("$", cmd, flush=True)
    full = "set -o pipefail; " + cmd + (f" 2>&1 | tee -a '{log}'" if log else "")
    r = subprocess.run(["bash", "-c", full], cwd=APP if os.path.isdir(APP) else W)
    if r.returncode:
        raise RuntimeError(f"Lệnh lỗi ({r.returncode}): {cmd}")

# 1. Chép code + bộ test vào /kaggle/working (Kaggle tự giải nén zip khi tạo dataset).
keep = f"{W}/_logs_keep"
if os.path.isdir(f"{APP}/logs"):
    shutil.rmtree(keep, ignore_errors=True)
    shutil.move(f"{APP}/logs", keep)
shutil.rmtree(f"{W}/AIM", ignore_errors=True)
src = [p for p in find("AIM/manifest.json") if json.load(open(p, encoding="utf-8")).get("variant") == "tpu"]
assert len(src) <= 1, f"Có nhiều hơn 1 gói TPU: {src}"
if src:
    shutil.copytree(os.path.dirname(src[0]), f"{W}/AIM")
else:
    zips = find("huhu-system-tpu.zip")
    assert len(zips) == 1, f"Cần đúng 1 dataset huhu-system-tpu, thấy: {zips}"
    shutil.unpack_archive(zips[0], W)
MANIFEST = json.load(open(f"{W}/AIM/manifest.json", encoding="utf-8"))
assert MANIFEST.get("variant") == "tpu", f"Gói không phải bản TPU: {MANIFEST.get('variant')}"
TAG = f"tpu {MANIFEST['created']}"
print("manifest:", MANIFEST["created"], len(MANIFEST["files"]), "file |", MANIFEST["vllm_tpu"])

In [ ]:
# 2. Kiểm tra liêm chính: mọi file code / dữ liệu / bộ test / LoRA adapter giống hệt bản local
bad = [arc for arc, h in MANIFEST["files"].items()
       if not os.path.exists(os.path.join(W, arc)) or hashlib.sha256(open(os.path.join(W, arc), "rb").read()).hexdigest() != h]
if bad:
    raise RuntimeError(f"{len(bad)} file khác bản local: {bad[:10]}")
print(f"OK: {len(MANIFEST['files'])} file khớp sha256 với bản local")
shutil.copyfile(f"{APP}/dot_env", f"{APP}/.env")  # .env được đóng gói dưới tên dot_env

# 3. Chạy tiếp: chỉ lấy kết quả của cùng gói (VARIANT.txt khớp), không trộn với lần chạy GPU/Ollama
if os.path.isdir(keep):
    shutil.copytree(keep, f"{APP}/logs", dirs_exist_ok=True)
for prev in find("AIM/Hybrid-database-main/logs/eval"):
    mark = os.path.join(prev, "VARIANT.txt")
    tag = open(mark, encoding="utf-8").read().strip() if os.path.exists(mark) else None
    if tag == TAG:
        shutil.copytree(os.path.dirname(prev), f"{APP}/logs", dirs_exist_ok=True)
        print("Chạy tiếp từ:", prev)
    else:
        print("Bỏ qua kết quả của gói khác:", prev, "| VARIANT.txt =", tag)
os.makedirs(f"{APP}/logs/eval", exist_ok=True)
mark = f"{APP}/logs/eval/VARIANT.txt"
if os.path.exists(mark):
    assert open(mark, encoding="utf-8").read().strip() == TAG, "logs/eval trong session thuộc gói khác"
open(mark, "w", encoding="utf-8").write(TAG + "\n")

In [ ]:
# 4. Hạ tầng (thay docker-compose): PostgreSQL 16 + pgvector, Redis, Neo4j 5.18; không cài Ollama
sh("SKIP_OLLAMA=1 bash serving/kaggle_services.sh", log=f"{W}/setup.log")
sh("df -h /tmp /kaggle/working | cat; free -g | cat; ls /dev | grep -i -E 'accel|vfio' | head")

In [ ]:
# 5. Thư viện Python cho hệ thống (chạy bằng Python mặc định của notebook; bge-m3 chạy CPU)
sh("pip install -q 'psycopg2-binary>=2.9.9' 'pgvector>=0.2.5' 'neo4j>=5.19.0' 'redis>=5.0.3' "
   "'sentence-transformers>=2.6.1' 'python-dotenv>=1.0.0' 'httpx>=0.27.0' 'fastapi>=0.110.0' 'pydantic>=2.6.0' "
   "'safetensors>=0.4.0' 'huggingface_hub>=0.23.0'")
if subprocess.run([sys.executable, "-c", "import torch"]).returncode:
    sh("pip install -q torch --index-url https://download.pytorch.org/whl/cpu")

# 6. vLLM cho TPU trong venv Python 3.12 riêng (bản ghim trong manifest)
if not os.path.exists(f"{VENV}/bin/python"):
    sh("pip install -q uv")
    sh(f"uv venv {VENV} --python 3.12")
sh(f"uv pip install --python {VENV}/bin/python -q '{MANIFEST['vllm_tpu']}'")
sh(f"{VENV}/bin/python -c 'import vllm, jax; print(\"vllm\", vllm.__version__); print(jax.devices())'")
sh(f"uv pip freeze --python {VENV}/bin/python > logs/eval/vllm_venv_freeze.txt")

In [ ]:
# 7. Model: Qwen2.5-7B-Instruct đúng revision; gộp LoRA (adapter đã kiểm sha256 ở bước 2) thành qwen-huhu bf16;
#    Gemma 3 từ Kaggle Models, không có thì tải từ Hugging Face đúng revision
from huggingface_hub import snapshot_download
os.makedirs(MODELS_DIR, exist_ok=True)
HF = MANIFEST["hf_models"]
PATHS = {}
PATHS["ext"] = snapshot_download(HF["ext"]["repo"], revision=HF["ext"]["revision"], cache_dir=MODELS_DIR,
                                 allow_patterns=["*.json", "*.safetensors", "*.txt"])
adapter = f"{W}/AIM/results/output/best_adapter"
PATHS["gen"] = f"{MODELS_DIR}/qwen-huhu"
if not os.path.exists(f"{PATHS['gen']}/config.json"):
    sh(f"python serving/tpu/merge_lora.py --base {PATHS['ext']} --adapter {adapter} --out {PATHS['gen']}")

def gemma(key, name):
    hits = [os.path.dirname(p) for p in find(f"{name}/**/config.json") + find(f"{name}/config.json")]
    if hits:
        print(f"{name}: Kaggle Models {hits[0]}")
        return hits[0], "kaggle_models"
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    p = snapshot_download(HF[key]["repo"], revision=HF[key]["revision"], cache_dir=MODELS_DIR, token=token,
                          allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"])
    return p, f"hf@{HF[key]['revision']}"

PATHS["check"], src_check = gemma("check", "gemma-3-12b-it")
PATHS["judge"], src_judge = gemma("judge", "gemma-3-4b-it")

def tree_sha(d):
    h = hashlib.sha256()
    for f in sorted(glob.glob(f"{d}/*.safetensors")):
        h.update(os.path.basename(f).encode())
        with open(f, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 24), b""):
                h.update(chunk)
    return h.hexdigest()

info = {k: {"path": p, "safetensors_sha256": tree_sha(p)} for k, p in PATHS.items()}
info["check"]["source"], info["judge"]["source"] = src_check, src_judge
json.dump(info, open(f"{APP}/logs/eval/models_tpu.json", "w"), indent=1)
print(json.dumps(info, indent=1))

In [ ]:
# 8. Bật 4 server vLLM (mỗi model 2 chip; vLLM biên dịch XLA lúc khởi động, có thể mất vài chục phút)
sys.path.insert(0, f"{APP}/serving/tpu")
import vllm_servers
sh(f"python serving/tpu/vllm_servers.py start --python {VENV}/bin/python "
   + " ".join(f"--{k} '{p}'" for k, p in PATHS.items()))
os.environ.update(vllm_servers.env_for_backend())  # backend và các script eval gọi đúng server của từng model
import httpx
for k, s in vllm_servers.SERVERS.items():
    r = httpx.post(f"{vllm_servers.url(k)}/chat/completions", timeout=600,
                   json={"model": s["name"], "messages": [{"role": "user", "content": "Chào bạn"}], "max_tokens": 20, "temperature": 0})
    r.raise_for_status()
    print(f"[{k}] {s['name']}: {r.json()['choices'][0]['message']['content']!r}")

In [ ]:
# 9. Embedding bge-m3 đúng revision đã dùng ở local (chạy CPU)
from huggingface_hub import hf_hub_download
B = MANIFEST["bge_m3"]
bge = snapshot_download("BAAI/bge-m3", revision=B["revision"],
                        allow_patterns=["*.json", "*.md", "1_Pooling/*", "sentencepiece.bpe.model"])
st = hf_hub_download("BAAI/bge-m3", "model.safetensors", revision=B["safetensors_revision"])
h = hashlib.sha256()
with open(st, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == B["safetensors_sha256"], f"bge-m3 model.safetensors khác bản local: {h.hexdigest()}"
if not os.path.exists(f"{bge}/model.safetensors"):
    os.symlink(os.path.realpath(st), f"{bge}/model.safetensors")
os.environ["EMBEDDING_MODEL_NAME"] = bge
os.environ["HF_HUB_OFFLINE"] = "1"
env = dict(l.split("=", 1) for l in open(f"{APP}/.env", encoding="utf-8-sig").read().splitlines() if "=" in l and not l.startswith("#"))
for k in ("RETRIEVER", "CBT_ENABLED", "GEN_MODEL", "EXT_MODEL", "CHECK_MODEL", "JUDGE_MODEL", "CHECK_ENABLED", "MAX_RETRIES"):
    print(f"{k} = {env.get(k, '(mặc định)')}")
for k in ("GEN_BASE_URL", "EXT_BASE_URL", "CHECK_BASE_URL", "JUDGE_BASE_URL"):
    print(f"{k} = {os.environ[k]}")

In [ ]:
# 10. Seed toàn bộ CSDL (PostgreSQL + pgvector, Neo4j) rồi kiểm số dòng so với file dữ liệu
if os.path.exists(f"{W}/seed.log"):
    os.remove(f"{W}/seed.log")
sh("python ingestion/seed_all.py", log=f"{W}/seed.log")
if "[LỖI" in open(f"{W}/seed.log", encoding="utf-8").read():
    raise RuntimeError("Seed lỗi, xem seed.log")
import psycopg2
d = f"{APP}/data/processed"
expect = {"nutrition_support": len(json.load(open(f"{d}/400 data dinh duong.json", encoding="utf-8"))),
          "state_effect_map": len(json.load(open(f"{d}/state_effect_map.json", encoding="utf-8"))["map"]),
          "cbt_catalog": len(json.load(open(f"{d}/cbt_catalog.json", encoding="utf-8"))["items"])}
conn = psycopg2.connect(dbname="mental_nutrition_db", user="admin", password="password123", host="localhost")
cur = conn.cursor()
for t, n in expect.items():
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    got = cur.fetchone()[0]
    print(f"{t}: {got} (file: {n})")
    assert got == n, t
cur.execute("SELECT COUNT(*), COUNT(embedding) FROM psychology_knowledge")
rows, emb = cur.fetchone()
n_psy = len(json.load(open(f"{d}/psychology_dataset_vi.json", encoding="utf-8")))
print("psychology_knowledge: dòng", rows, "có embedding", emb, "(file:", n_psy, ")")
assert rows == emb == n_psy
conn.close()
from neo4j import GraphDatabase
with GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123")) as drv:
    c = drv.execute_query("MATCH (n) RETURN count(n) AS c").records[0]["c"]
print("Neo4j nodes:", c)
assert c > 0

In [ ]:
# 11. Truy xuất qua PostgreSQL phải cho cùng đích tác động với bản đọc file JSON, với mọi tổ hợp state
os.chdir(APP)
sys.path.insert(0, APP)
from itertools import product
from backend.constants import STATES, STATE_LABELS
from retrieval.state_retriever import OfflineStateRetriever, StateRetriever
off, pg = OfflineStateRetriever(), StateRetriever()
n = 0
for vals in product(*[[None] + list(STATE_LABELS[s]) for s in STATES]):
    keys = {f"{s}:{v}" for s, v in zip(STATES, vals) if v}
    a, b = off.target_weights(keys), pg.target_weights(keys)
    assert a.keys() == b.keys() and all(abs(a[k] - b[k]) < 1e-4 for k in a), (keys, a, b)
    n += 1
print(f"OK: {n} tổ hợp state, PostgreSQL == JSON")

In [ ]:
# 12. Unit test (không cần model)
sh("python -m unittest discover -s tests", log=f"{W}/unittest.log")

In [ ]:
# 13. Chạy thử toàn hệ thống với câu tự soạn (không lấy từ bộ test)
from backend import config
from backend.pipeline import build_default_pipeline
assert config.RETRIEVER == "postgres" and config.CHECK_MODEL == "gemma3:12b" and config.JUDGE_MODEL == "gemma3:4b"
assert config.CHECK_BASE_URL != config.JUDGE_BASE_URL
p = build_default_pipeline()
for msg in ["Dạo này mình hơi mệt.",
            "Mình đang uống thuốc ngủ bác sĩ kê, có nên ăn gì thêm không?",
            "Tim mình đập loạn, thở không nổi, đang ngồi bệt ngoài hành lang."]:
    t = time.time()
    r = p.handle(f"smoke-{time.time()}", msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:200]}\n")
sid = f"smoke-multi-{time.time()}"
for msg in ["Mình căng thẳng vì công việc, tối nào cũng mất ngủ.", "Mình ăn ít hẳn đi, tâm trạng thì buồn chán."]:
    t = time.time()
    r = p.handle(sid, msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:300]}\n")

In [ ]:
# 14. LLM #3 trên 36 ca tự soạn có đáp án (scripts/judge_bench.py, không lấy từ bộ test): xem trước khi chạy test
sh("python -u scripts/judge_bench.py gemma3:12b", log=f"{APP}/logs/eval/run_judge_bench_llm3.txt")

## Chạy bộ test
- **Crisis** (164 ca: tự sát, tự hại, hoảng loạn cấp, một nhóm khủng hoảng) và **Retrieval**: LLM #1 và truy xuất PostgreSQL.
- **P0** (hệ thống cũ, bản trong báo cáo lần trước, chạy bằng code cũ), **P, B1, B2, B3** trên crisis (164) và safety (300): pha 1 sinh câu trả lời, pha 2 chấm bằng giám khảo `gemma3:4b`.
- **Grounding và định tuyến nhiều lượt** (tập test fine-tune B1–B4), trace ghi riêng `logs/turns_grounding.jsonl`.

Dừng giữa chừng thì gắn Output làm input và chạy lại notebook: ca đã xong được bỏ qua.

In [ ]:
if not glob.glob(f"{APP}/logs/eval/crisis_*.jsonl"):
    sh("python -u scripts/eval_system.py crisis", log=f"{APP}/logs/eval/run_crisis_kaggle.txt")
if not glob.glob(f"{APP}/logs/eval/retrieval_*.jsonl"):
    sh("python -u scripts/eval_system.py retrieval", log=f"{APP}/logs/eval/run_retrieval_kaggle.txt")

In [ ]:
# Hệ thống đề xuất trước, rồi 2 baseline trả lời trực tiếp, cuối cùng pipeline + model gốc
for s in ["P", "B1", "B2", "B3"]:
    sh(f"python -u scripts/eval_baselines.py gen --systems {s} --suites crisis safety --workers 4",
       log=f"{APP}/logs/eval/run_gen_{s}.txt")
# P0 = hệ thống cũ (bản trong báo cáo lần trước, commit trong manifest), baseline của hệ thống final.
# Chạy bằng chính code cũ (AIM/p_old, đã kiểm sha256 ở bước 2), cùng 4 server, cùng bộ test. Chỉ thay backend/llm.py
# bằng bản mới: client HTTP thêm dự phòng khi server không hỗ trợ JSON mode, không đổi logic hệ thống.
OLD = f"{W}/AIM/p_old/Hybrid-database-main"
for link, target in ((f"{OLD}/data", f"{APP}/data"), (f"{OLD}/.env", f"{APP}/.env"), (f"{W}/AIM/p_old/data", f"{W}/AIM/data")):
    if not os.path.lexists(link):
        os.symlink(target, link)
shutil.copyfile(f"{APP}/backend/llm.py", f"{OLD}/backend/llm.py")
old_out, new_out = f"{OLD}/logs/eval/baselines", f"{APP}/logs/eval/baselines"
os.makedirs(old_out, exist_ok=True)
for su in ("crisis", "safety"):  # chạy tiếp: đưa kết quả P0 đã có vào cây code cũ
    if os.path.exists(f"{new_out}/P0_{su}_replies.jsonl"):
        shutil.copyfile(f"{new_out}/P0_{su}_replies.jsonl", f"{old_out}/P_{su}_replies.jsonl")
sh(f"cd {OLD} && TURN_LOG_PATH={APP}/logs/turns_P0.jsonl python -u scripts/eval_baselines.py gen --systems P "
   "--suites crisis safety --workers 4", log=f"{APP}/logs/eval/run_gen_P0.txt")
for su in ("crisis", "safety"):
    shutil.copyfile(f"{old_out}/P_{su}_replies.jsonl", f"{new_out}/P0_{su}_replies.jsonl")
sh("python -u scripts/eval_baselines.py score --systems P0 P B1 B2 B3 --suites crisis safety",
   log=f"{APP}/logs/eval/run_score.txt")
sh("python -u scripts/eval_baselines.py report", log=f"{APP}/logs/eval/run_report_kaggle.txt")
sh(f"python -u scripts/analyze_final.py {APP}", log=f"{APP}/logs/eval/run_analyze_final.txt")

In [ ]:
G = f"TURN_LOG_PATH={APP}/logs/turns_grounding.jsonl"
sh(f"{G} python -u scripts/eval_grounding.py gen-a --workers 4", log=f"{APP}/logs/eval/run_grounding_gen_a.txt")
sh(f"{G} python -u scripts/eval_grounding.py gen-b --workers 4", log=f"{APP}/logs/eval/run_grounding_gen_b.txt")
sh(f"{G} python -u scripts/eval_grounding.py score", log=f"{APP}/logs/eval/run_grounding_score.txt")
sh(f"{G} python -u scripts/eval_grounding.py report", log=f"{APP}/logs/eval/run_grounding_report.txt")

In [ ]:
# Nén kết quả để tải về (logs/eval: jsonl từng ca + bảng tổng hợp; logs/turns*: trace từng lượt; log vLLM)
shutil.copytree(vllm_servers.LOG_DIR, f"{APP}/logs/vllm", dirs_exist_ok=True)
shutil.make_archive(f"{W}/eval_tpu_results", "zip", APP, "logs")
print(f"{W}/eval_tpu_results.zip", round(os.path.getsize(f"{W}/eval_tpu_results.zip") / 1e6, 1), "MB")
sh("python serving/tpu/vllm_servers.py stop")